# 08d · Export the test text for the champion's final predictions: BankSight


**What this does:** loads the libraries.

In [1]:
from pathlib import Path

import duckdb
import joblib
import pandas as pd

**What this does:** sets the file locations. The export goes to `data/colab/`, next to the earlier files.

In [2]:
ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
WAREHOUSE = ROOT / "data" / "warehouse" / "banksight.duckdb"
EXPORT_DIR = ROOT / "data" / "colab"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PII_FILE = ROOT / "models" / "07_pii_patterns.joblib"

assert WAREHOUSE.exists(), "Run notebooks 01 and 01b first."
assert PII_FILE.exists(), "Run notebook 07 first."

MAX_CHARS = 1800   # same cut-off as in 08a

**What this does:** reads the test-period complaints that have a final outcome. We select **only the id, product, issue and text**. The outcome column is not selected.

In [3]:
con = duckdb.connect(str(WAREHOUSE), read_only=True)
df = con.sql("""
    SELECT l.complaint_id, l.date_received, l.product_group, r.issue, r.narrative
    FROM features.complaint_labels l
    JOIN raw.cfpb_complaints r USING (complaint_id)
    WHERE l.target_eligible
      AND l.split = 'test'
      AND r.narrative IS NOT NULL AND length(trim(r.narrative)) > 0
""").df()
con.close()

df["issue"] = df["issue"].fillna("Unknown").astype(str)
df = df.sort_values("date_received", kind="stable").reset_index(drop=True)
print(f"test complaints: {len(df):,} ({df['date_received'].min()} to {df['date_received'].max()})")

test complaints: 26,452 (2025-11-01 00:00:00 to 2026-07-29 00:00:00)


**What this does:** redacts personal-data patterns with the same patterns as 08a, then builds the same `product | issue | narrative` text.

In [4]:
pii_patterns = joblib.load(PII_FILE)

text = df["narrative"]
for name, pattern in pii_patterns.items():
    text = text.str.replace(pattern, f"[{name.upper()}]", regex=True)

df["input_text"] = df["product_group"] + " | " + df["issue"] + " | " + text.str.slice(0, MAX_CHARS)
print(df["input_text"].iloc[0][:300])

Bank account | Managing an account | XXXX. I was defrauded {$280000.00} by an imposter pretending to be the prince of XXXX, on the XXXX  platform, I was enticed into investing my {$280000.00} which was the proceeds from the sale of my mobile home in XXXX  into a fictitious bank with a high yield int


**What this does:** saves `test.parquet` with only three columns. Check that there is no label column.

In [5]:
test_export = df[["complaint_id", "date_received", "input_text"]]
test_export.to_parquet(EXPORT_DIR / "test.parquet", index=False)

print("columns in the file:", list(test_export.columns))
print(f"test.parquet: {(EXPORT_DIR / 'test.parquet').stat().st_size / 1e6:.1f} MB, {len(test_export):,} rows")

columns in the file: ['complaint_id', 'date_received', 'input_text']
test.parquet: 15.4 MB, 26,452 rows


## What to do next
1. Upload `data/colab/test.parquet` to Google Drive, into the same **`banksight_colab`** folder as before.
2. Open **08e_colab_predict_test.ipynb** on Colab (T4 GPU) and run it. It needs the `roberta_relief` folder that 08b saved in `banksight_colab/outputs/`.
3. Download the file it creates, `test_predictions_roberta.csv`, into your laptop's `data/colab/outputs/` folder.

## How to explain this in an interview
> "The test set is sealed. I sent only the text to the GPU, never the outcomes, so nobody could tune anything against the test answers."